### Numerical Prediction

**Requirements (advanced dataset)**
1. Two different types of regression algorithm:
   - **Model A**: non-temporal → XGBoost Regressor
   - **Model B**: inherently temporal → LSTM Regressor (PyTorch)
2. Describe similar details as classification (2A)
3. Highlight differences between classification and regression tasks


#### 1. Configuration & Loading

In [2]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)

OUTPUT_DIR = Path("../outputs")
FIG_DIR    = Path("../figures")

# Load the 1C modelling dataset
df = pd.read_csv(OUTPUT_DIR / "data_modelling.csv",
                 parse_dates=["date", "target_date"])
df = df.sort_values(["date", "id"]).reset_index(drop=True)

# Target is continuous next-day mood (no discretization)
target_col = "target_next_day_mood"

print(f"Loaded {len(df)} instances, {df['id'].nunique()} patients")
print(f"Target stats:")
print(df[target_col].describe().to_string())

Loaded 1394 instances, 27 patients
Target stats:
count    1394.000000
mean        6.995744
std         0.722920
min         3.000000
25%         6.600000
50%         7.000000
75%         7.500000
max         9.333333


#### 2. Walk-Forward Temporal CV

Reuse the identical fold structure from the classification step for direct comparability.

In [3]:
unique_dates = sorted(df["date"].unique())
n_dates = len(unique_dates)
block_size = n_dates // 5

date_blocks = []
for i in range(5):
    start = i * block_size
    end = (i + 1) * block_size if i < 4 else n_dates
    date_blocks.append(set(unique_dates[start:end]))

cv_folds = []
for k in range(4):
    train_dates = set()
    for j in range(k + 1):
        train_dates |= date_blocks[j]
    test_dates = date_blocks[k + 1]
    train_idx = df[df["date"].isin(train_dates)].index.tolist()
    test_idx  = df[df["date"].isin(test_dates)].index.tolist()
    cv_folds.append((train_idx, test_idx))

print("Walk-forward CV folds:")
for i, (tr, te) in enumerate(cv_folds):
    print(f"  Fold {i+1}: train {len(tr):4d} │ test {len(te):4d}")

Walk-forward CV folds:
  Fold 1: train  203 │ test  395
  Fold 2: train  598 │ test  490
  Fold 3: train 1088 │ test   67
  Fold 4: train 1155 │ test  239


#### 3. Evaluation Helpers & Baselines

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regression(y_true, y_pred, fold_name=""):
    """Compute regression metrics for one fold."""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    return {"fold": fold_name, "mae": mae, "mse": mse, "r2": r2,
            "n_test": len(y_true)}

def print_reg_summary(results, model_name):
    """Print CV summary."""
    maes = [r["mae"] for r in results]
    mses = [r["mse"] for r in results]
    r2s  = [r["r2"] for r in results]
    print(f"\n{'='*60}")
    print(f"{model_name} — CV Summary")
    print(f"{'='*60}")
    for r in results:
        print(f"  {r['fold']}: MAE={r['mae']:.4f}  MSE={r['mse']:.4f}  "
              f"R²={r['r2']:.4f}  (n={r['n_test']})")
    print(f"  {'─'*50}")
    print(f"  Mean MAE = {np.mean(maes):.4f} ± {np.std(maes):.4f}")
    print(f"  Mean MSE = {np.mean(mses):.4f} ± {np.std(mses):.4f}")
    print(f"  Mean R²  = {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
    return np.mean(maes), np.mean(mses), np.mean(r2s)

##### Baselines

Two simple baselines from the EDA (b_09) to establish the floor:
- **Naive**: predict today's mood for tomorrow (carry-forward)
- **Rolling w=5**: predict the 5-day rolling mean of past mood

In [5]:
baseline_models = {}

for bname, shift_or_window in [("naive", 1), ("rolling_w5", 5)]:
    results = []
    all_true, all_pred = [], []

    for i, (train_idx, test_idx) in enumerate(cv_folds):
        test = df.loc[test_idx].copy()

        if bname == "naive":
            # Carry-forward: predict mood_lag1 (today's mood for tomorrow)
            preds = test["mood_lag1"].values
        else:
            # Rolling w=7 mean
            preds = test["mood_w5_mean"].values  # w5 is our window mean

        # Drop instances where baseline prediction is NaN
        valid = ~np.isnan(preds)
        y_true = test[target_col].values[valid]
        y_pred = preds[valid]

        if len(y_true) > 0:
            results.append(evaluate_regression(y_true, y_pred, f"Fold {i+1}"))
            all_true.extend(y_true)
            all_pred.extend(y_pred)

    bm, bms, br2 = print_reg_summary(results, f"Baseline: {bname}")
    baseline_models[bname] = {"mae": bm, "mse": bms, "r2": br2,
                              "all_true": all_true, "all_pred": all_pred}


Baseline: naive — CV Summary
  Fold 1: MAE=0.4365  MSE=0.3938  R²=0.2889  (n=356)
  Fold 2: MAE=0.5121  MSE=0.5479  R²=0.0349  (n=486)
  Fold 3: MAE=0.6232  MSE=0.6794  R²=-0.5420  (n=59)
  Fold 4: MAE=0.6154  MSE=0.5999  R²=-1.4579  (n=54)
  ──────────────────────────────────────────────────
  Mean MAE = 0.5468 ± 0.0773
  Mean MSE = 0.5552 ± 0.1043
  Mean R²  = -0.4190 ± 0.6711

Baseline: rolling_w5 — CV Summary
  Fold 1: MAE=0.4802  MSE=0.4445  R²=0.2618  (n=276)
  Fold 2: MAE=0.4773  MSE=0.4373  R²=0.2339  (n=483)
  Fold 3: MAE=0.5363  MSE=0.4655  R²=-0.0834  (n=47)
  ──────────────────────────────────────────────────
  Mean MAE = 0.4979 ± 0.0272
  Mean MSE = 0.4491 ± 0.0119
  Mean R²  = 0.1374 ± 0.1566


#### 4. Model A - XGBoost Regressor

Same rationale as the classification step: handles NaN natively, uses all 1,394 instances,
provides feature importance. Hyperparameters tuned via RandomizedSearchCV
with the same temporal CV folds.

In [6]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

meta_cols = ["id", "date", "target_date", "target_next_day_mood",
             "quality_tier"]
feat_cols = [c for c in df.columns if c not in meta_cols]

X = df[feat_cols].values
y = df[target_col].values

print(f"Feature matrix: {X.shape}")

Feature matrix: (1394, 23)


##### 4a. Hyperparameter Search

In [7]:
param_dist = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [3, 4, 5, 6, 8],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2],
    "subsample":         [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree":  [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight":  [1, 3, 5, 7],
    "gamma":             [0, 0.1, 0.2, 0.5],
    "reg_alpha":         [0, 0.01, 0.1, 1.0],
    "reg_lambda":        [0.5, 1.0, 2.0, 5.0],
}

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    verbosity=0,
)

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=60,
    scoring="neg_mean_absolute_error",
    cv=cv_folds,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=False,
)

search.fit(X, y)
best_params = search.best_params_
print(f"\nBest hyperparameters (neg-MAE = {search.best_score_:.4f}):")
for k, v in sorted(best_params.items()):
    print(f"  {k}: {v}")

Fitting 4 folds for each of 60 candidates, totalling 240 fits

Best hyperparameters (neg-MAE = -0.4877):
  colsample_bytree: 0.7
  gamma: 0.5
  learning_rate: 0.05
  max_depth: 3
  min_child_weight: 7
  n_estimators: 500
  reg_alpha: 1.0
  reg_lambda: 5.0
  subsample: 1.0


##### 4b. Per-Fold Evaluation

In [8]:
xgb_results = []
xgb_all_true = []
xgb_all_pred = []
xgb_all_errors = []  # store individual errors for Task 5B

for i, (train_idx, test_idx) in enumerate(cv_folds):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = XGBRegressor(**best_params, objective="reg:squarederror",
                         random_state=42, verbosity=0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    xgb_results.append(evaluate_regression(y_test, y_pred, f"Fold {i+1}"))
    xgb_all_true.extend(y_test)
    xgb_all_pred.extend(y_pred)
    xgb_all_errors.extend(y_test - y_pred)

xgb_mae, xgb_mse, xgb_r2 = print_reg_summary(xgb_results, "XGBoost Regressor")

# Pooled metrics
print(f"\nPooled metrics (XGBoost):")
print(f"  MAE = {mean_absolute_error(xgb_all_true, xgb_all_pred):.4f}")
print(f"  MSE = {mean_squared_error(xgb_all_true, xgb_all_pred):.4f}")
print(f"  R²  = {r2_score(xgb_all_true, xgb_all_pred):.4f}")


XGBoost Regressor — CV Summary
  Fold 1: MAE=0.5148  MSE=0.4834  R²=0.1118  (n=395)
  Fold 2: MAE=0.4510  MSE=0.3871  R²=0.3159  (n=490)
  Fold 3: MAE=0.4964  MSE=0.4716  R²=0.0224  (n=67)
  Fold 4: MAE=0.4886  MSE=0.4478  R²=-0.0481  (n=239)
  ──────────────────────────────────────────────────
  Mean MAE = 0.4877 ± 0.0232
  Mean MSE = 0.4475 ± 0.0371
  Mean R²  = 0.1005 ± 0.1367

Pooled metrics (XGBoost):
  MAE = 0.4823
  MSE = 0.4360
  R²  = 0.1771


##### 4c. Feature Importance

In [9]:
final_xgb = XGBRegressor(**best_params, objective="reg:squarederror",
                          random_state=42, verbosity=0)
final_xgb.fit(X, y)

importances = pd.Series(final_xgb.feature_importances_, index=feat_cols)
importances = importances.sort_values(ascending=False)

print("Feature importance — regression (top 10):")
for fname, imp in importances.head(10).items():
    print(f"  {fname:40s}  {imp:.4f}")

importances.to_csv(OUTPUT_DIR / "xgb_regression_feature_importance.csv")

Feature importance — regression (top 10):
  mood_w5_mean                              0.2597
  mood_lag1                                 0.1539
  mood_lag2                                 0.0419
  mood_w5_std                               0.0415
  circumplex_valence_w5_std                 0.0353
  appCat_communication_lag1                 0.0331
  activity_w5_std                           0.0319
  screen_w5_mean                            0.0310
  activity_w5_mean                          0.0297
  dow_cos                                   0.0290


#### 5. Model B — LSTM Regressor

Same LSTM architecture as the classification step but with a single linear output
(no softmax, no class weights). MSE loss for training.

##### 5a. Sequence Preparation

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

df_daily = pd.read_csv(OUTPUT_DIR / "data_imputed.csv", parse_dates=["date"])
df_daily = df_daily.sort_values(["id", "date"]).reset_index(drop=True)
df_daily["activity_t"] = np.sqrt(df_daily["activity"])

SEQ_FEATURES = ["mood", "circumplex.valence", "activity_t", "screen",
                "appCat.communication"]
W = 5

df_daily_indexed = df_daily.set_index(["id", "date"])[SEQ_FEATURES]

sequences = []
targets_seq = []
instance_dates = []
instance_indices = []

for idx, row in df.iterrows():
    pid = row["id"]
    end_date = row["date"]
    target_val = row[target_col]

    seq_dates = pd.date_range(end=end_date, periods=W, freq="D")
    try:
        seq = df_daily_indexed.loc[[(pid, d) for d in seq_dates]].values
    except KeyError:
        continue
    if np.isnan(seq).any():
        continue

    sequences.append(seq)
    targets_seq.append(target_val)
    instance_dates.append(end_date)
    instance_indices.append(idx)

sequences = np.array(sequences, dtype=np.float32)
targets_seq = np.array(targets_seq, dtype=np.float32)
instance_dates = np.array(instance_dates)
instance_indices = np.array(instance_indices)

print(f"Complete sequences: {len(sequences)} / {len(df)} "
      f"({len(sequences)/len(df)*100:.1f}%)")
print(f"Sequence shape: {sequences.shape}")

Complete sequences: 660 / 1394 (47.3%)
Sequence shape: (660, 5, 5)


##### 5b. LSTM Temporal CV Folds

In [11]:
seq_dates_unique = sorted(set(instance_dates))
n_seq_dates = len(seq_dates_unique)
lstm_block_size = n_seq_dates // 4

lstm_date_blocks = []
for i in range(4):
    start = i * lstm_block_size
    end = (i + 1) * lstm_block_size if i < 3 else n_seq_dates
    lstm_date_blocks.append(set(seq_dates_unique[start:end]))

lstm_cv_folds = []
for k in range(3):
    train_dates = set()
    for j in range(k + 1):
        train_dates |= lstm_date_blocks[j]
    test_dates = lstm_date_blocks[k + 1]
    train_seq_idx = [i for i, d in enumerate(instance_dates) if d in train_dates]
    test_seq_idx  = [i for i, d in enumerate(instance_dates) if d in test_dates]
    lstm_cv_folds.append((train_seq_idx, test_seq_idx))

print("LSTM walk-forward CV folds:")
for fi, (tr, te) in enumerate(lstm_cv_folds):
    tr_d = instance_dates[tr] if len(tr) else []
    te_d = instance_dates[te] if len(te) else []
    tr_min = pd.Timestamp(min(tr_d)).strftime('%Y-%m-%d') if len(tr_d) else 'N/A'
    tr_max = pd.Timestamp(max(tr_d)).strftime('%Y-%m-%d') if len(tr_d) else 'N/A'
    te_min = pd.Timestamp(min(te_d)).strftime('%Y-%m-%d') if len(te_d) else 'N/A'
    te_max = pd.Timestamp(max(te_d)).strftime('%Y-%m-%d') if len(te_d) else 'N/A'
    print(f"  Fold {fi+1}: train {len(tr):4d} ({tr_min}–{tr_max}) │ "
          f"test {len(te):4d} ({te_min}–{te_max})")

LSTM walk-forward CV folds:
  Fold 1: train  130 (2014-03-21–2014-03-31) │ test  243 (2014-04-01–2014-04-23)
  Fold 2: train  373 (2014-03-21–2014-04-23) │ test  246 (2014-04-24–2014-05-04)
  Fold 3: train  619 (2014-03-21–2014-05-04) │ test   41 (2014-05-17–2014-05-30)


##### 5c. LSTM Regressor Architecture & Training

In [12]:
class MoodLSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, (h_n, _) = self.lstm(x)
        out = self.dropout(h_n[-1])
        out = self.fc(out).squeeze(-1)  # (batch,)
        return out


class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def train_lstm_reg_fold(X_train, y_train, X_test, y_test,
                        hidden_size=32, lr=0.001, dropout=0.2,
                        epochs=80, batch_size=32, patience=15):
    """Train LSTM regressor on one fold, return predictions."""
    n_train, seq_len, n_feat = X_train.shape
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(
        X_train.reshape(-1, n_feat)).reshape(n_train, seq_len, n_feat)
    X_te_s = scaler.transform(
        X_test.reshape(-1, n_feat)).reshape(X_test.shape[0], seq_len, n_feat)

    # Standardize target too (helps training stability)
    y_mean, y_std = y_train.mean(), y_train.std()
    y_train_s = (y_train - y_mean) / y_std

    train_ds = SequenceDataset(X_tr_s, y_train_s)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MoodLSTMRegressor(input_size=n_feat, hidden_size=hidden_size,
                               dropout=dropout).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5)

    best_loss = float("inf")
    patience_counter = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            preds = model(X_b)
            loss = criterion(preds, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(y_b)

        epoch_loss /= len(train_ds)
        scheduler.step(epoch_loss)

        if epoch_loss < best_loss - 1e-4:
            best_loss = epoch_loss
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_te_s, dtype=torch.float32).to(device)
        y_pred_s = model(X_t).cpu().numpy()

    # De-standardize predictions
    y_pred = y_pred_s * y_std + y_mean
    return y_pred, epoch + 1

##### 5d. LSTM Hyperparameter Search

In [13]:
lstm_configs = [
    {"hidden_size": 32,  "lr": 0.001, "dropout": 0.2},
    {"hidden_size": 32,  "lr": 0.005, "dropout": 0.3},
    {"hidden_size": 64,  "lr": 0.001, "dropout": 0.3},
    {"hidden_size": 64,  "lr": 0.005, "dropout": 0.2},
    {"hidden_size": 128, "lr": 0.001, "dropout": 0.3},
]

print(f"LSTM regression HP search: {len(lstm_configs)} configs "
      f"× {len(lstm_cv_folds)} folds\n")

best_lstm_mae = float("inf")
best_lstm_config = None

for ci, config in enumerate(lstm_configs):
    fold_maes = []
    for fi, (tr_idx, te_idx) in enumerate(lstm_cv_folds):
        if len(tr_idx) == 0 or len(te_idx) == 0:
            continue
        y_pred, _ = train_lstm_reg_fold(
            sequences[tr_idx], targets_seq[tr_idx],
            sequences[te_idx], targets_seq[te_idx], **config)
        mae = mean_absolute_error(targets_seq[te_idx], y_pred)
        fold_maes.append(mae)

    mean_mae = np.mean(fold_maes) if fold_maes else float("inf")
    print(f"  Config {ci+1}: h={config['hidden_size']:3d}  "
          f"lr={config['lr']:.3f}  drop={config['dropout']:.1f}  "
          f"→ mean MAE = {mean_mae:.4f}")

    if mean_mae < best_lstm_mae:
        best_lstm_mae = mean_mae
        best_lstm_config = config

print(f"\nBest LSTM config: {best_lstm_config}  (MAE = {best_lstm_mae:.4f})")

LSTM regression HP search: 5 configs × 3 folds

  Config 1: h= 32  lr=0.001  drop=0.2  → mean MAE = 0.4933
  Config 2: h= 32  lr=0.005  drop=0.3  → mean MAE = 0.6238
  Config 3: h= 64  lr=0.001  drop=0.3  → mean MAE = 0.4854
  Config 4: h= 64  lr=0.005  drop=0.2  → mean MAE = 0.6509
  Config 5: h=128  lr=0.001  drop=0.3  → mean MAE = 0.5431

Best LSTM config: {'hidden_size': 64, 'lr': 0.001, 'dropout': 0.3}  (MAE = 0.4854)


##### 5e. Final LSTM Evaluation

In [14]:
torch.manual_seed(42)
np.random.seed(42)

lstm_results = []
lstm_all_true = []
lstm_all_pred = []
lstm_all_errors = []

for i, (tr_idx, te_idx) in enumerate(lstm_cv_folds):
    if len(tr_idx) == 0 or len(te_idx) == 0:
        print(f"  Fold {i+1}: SKIPPED (empty)")
        continue

    y_pred, n_epochs = train_lstm_reg_fold(
        sequences[tr_idx], targets_seq[tr_idx],
        sequences[te_idx], targets_seq[te_idx], **best_lstm_config)

    y_te = targets_seq[te_idx]
    lstm_results.append(evaluate_regression(y_te, y_pred, f"Fold {i+1}"))
    lstm_all_true.extend(y_te)
    lstm_all_pred.extend(y_pred)
    lstm_all_errors.extend(y_te - y_pred)
    print(f"  Fold {i+1}: MAE={lstm_results[-1]['mae']:.4f}  "
          f"MSE={lstm_results[-1]['mse']:.4f}  "
          f"(trained {n_epochs} epochs, n={len(y_te)})")

lstm_mae, lstm_mse, lstm_r2 = print_reg_summary(lstm_results, "LSTM Regressor")

print(f"\nPooled metrics (LSTM):")
print(f"  MAE = {mean_absolute_error(lstm_all_true, lstm_all_pred):.4f}")
print(f"  MSE = {mean_squared_error(lstm_all_true, lstm_all_pred):.4f}")
print(f"  R²  = {r2_score(lstm_all_true, lstm_all_pred):.4f}")

  Fold 1: MAE=0.4789  MSE=0.4447  (trained 80 epochs, n=243)
  Fold 2: MAE=0.4199  MSE=0.3190  (trained 80 epochs, n=246)
  Fold 3: MAE=0.5897  MSE=0.5409  (trained 80 epochs, n=41)

LSTM Regressor — CV Summary
  Fold 1: MAE=0.4789  MSE=0.4447  R²=0.1697  (n=243)
  Fold 2: MAE=0.4199  MSE=0.3190  R²=0.4032  (n=246)
  Fold 3: MAE=0.5897  MSE=0.5409  R²=-0.2372  (n=41)
  ──────────────────────────────────────────────────
  Mean MAE = 0.4961 ± 0.0704
  Mean MSE = 0.4349 ± 0.0908
  Mean R²  = 0.1119 ± 0.2646

Pooled metrics (LSTM):
  MAE = 0.4601
  MSE = 0.3938
  R²  = 0.2677


#### 6. Model Comparison

In [15]:
print(f"\n{'='*70}")
print("Model Comparison — Regression")
print(f"{'='*70}")
print(f"  {'Model':<25s}  {'MAE':>8s}  {'MSE':>8s}  {'R²':>8s}  {'N':>6s}")
print(f"  {'─'*60}")

# Baselines
for bname in ["naive", "rolling_w5"]:
    b = baseline_models[bname]
    print(f"  {bname:<25s}  {b['mae']:>8.4f}  {b['mse']:>8.4f}  "
          f"{b['r2']:>8.4f}  {'–':>6s}")

# Models (mean of folds)
print(f"  {'XGBoost':<25s}  {xgb_mae:>8.4f}  {xgb_mse:>8.4f}  "
      f"{xgb_r2:>8.4f}  {'1394':>6s}")
print(f"  {'LSTM':<25s}  {lstm_mae:>8.4f}  {lstm_mse:>8.4f}  "
      f"{lstm_r2:>8.4f}  {'660':>6s}")

# Pooled
xgb_pooled_mae = mean_absolute_error(xgb_all_true, xgb_all_pred)
xgb_pooled_mse = mean_squared_error(xgb_all_true, xgb_all_pred)
xgb_pooled_r2  = r2_score(xgb_all_true, xgb_all_pred)
lstm_pooled_mae = mean_absolute_error(lstm_all_true, lstm_all_pred)
lstm_pooled_mse = mean_squared_error(lstm_all_true, lstm_all_pred)
lstm_pooled_r2  = r2_score(lstm_all_true, lstm_all_pred)

print(f"\n  Pooled metrics:")
print(f"  {'XGBoost (pooled)':<25s}  {xgb_pooled_mae:>8.4f}  "
      f"{xgb_pooled_mse:>8.4f}  {xgb_pooled_r2:>8.4f}  {'1191':>6s}")
print(f"  {'LSTM (pooled)':<25s}  {lstm_pooled_mae:>8.4f}  "
      f"{lstm_pooled_mse:>8.4f}  {lstm_pooled_r2:>8.4f}  {'530':>6s}")


Model Comparison — Regression
  Model                           MAE       MSE        R²       N
  ────────────────────────────────────────────────────────────
  naive                        0.5468    0.5552   -0.4190       –
  rolling_w5                   0.4979    0.4491    0.1374       –
  XGBoost                      0.4877    0.4475    0.1005    1394
  LSTM                         0.4961    0.4349    0.1119     660

  Pooled metrics:
  XGBoost (pooled)             0.4823    0.4360    0.1771    1191
  LSTM (pooled)                0.4601    0.3938    0.2677     530


#### 7. Error Analysis for Metric Comparison

Compute per-instance error statistics needed for Metric Sensitivity Analysis
(impact of MAE vs MSE on model evaluation).

In [16]:
xgb_errors = np.array(xgb_all_errors)
lstm_errors = np.array(lstm_all_errors)

for name, errors in [("XGBoost", xgb_errors), ("LSTM", lstm_errors)]:
    abs_err = np.abs(errors)
    sq_err  = errors ** 2
    n = len(errors)

    # Top 5% of errors by magnitude
    threshold = np.percentile(abs_err, 95)
    top5_mask = abs_err >= threshold

    mae_total = abs_err.sum()
    mse_total = sq_err.sum()
    mae_top5_share = abs_err[top5_mask].sum() / mae_total * 100
    mse_top5_share = sq_err[top5_mask].sum() / mse_total * 100

    print(f"\n{name} — Error distribution:")
    print(f"  Mean |error|     = {abs_err.mean():.4f}")
    print(f"  Median |error|   = {np.median(abs_err):.4f}")
    print(f"  Max |error|      = {abs_err.max():.4f}")
    print(f"  Std of errors    = {errors.std():.4f}")
    print(f"  Top 5% threshold = {threshold:.4f}")
    print(f"  Top 5% share of MAE = {mae_top5_share:.1f}%  "
          f"(proportional would be 5%)")
    print(f"  Top 5% share of MSE = {mse_top5_share:.1f}%  "
          f"(proportional would be 5%)")
    print(f"  MSE/MAE sensitivity ratio = {mse_top5_share/mae_top5_share:.2f}×")

# Save error arrays for Task 5B
error_df = pd.DataFrame({
    "y_true": xgb_all_true,
    "y_pred_xgb": xgb_all_pred,
    "error_xgb": xgb_all_errors,
})
error_df.to_csv(OUTPUT_DIR / "regression_errors_xgb.csv", index=False)

error_df_lstm = pd.DataFrame({
    "y_true": lstm_all_true,
    "y_pred_lstm": lstm_all_pred,
    "error_lstm": lstm_all_errors,
})
error_df_lstm.to_csv(OUTPUT_DIR / "regression_errors_lstm.csv", index=False)


XGBoost — Error distribution:
  Mean |error|     = 0.4823
  Median |error|   = 0.3699
  Max |error|      = 3.4948
  Std of errors    = 0.6603
  Top 5% threshold = 1.2467
  Top 5% share of MAE = 18.6%  (proportional would be 5%)
  Top 5% share of MSE = 40.5%  (proportional would be 5%)
  MSE/MAE sensitivity ratio = 2.17×

LSTM — Error distribution:
  Mean |error|     = 0.4601
  Median |error|   = 0.3494
  Max |error|      = 3.4107
  Std of errors    = 0.6274
  Top 5% threshold = 1.2221
  Top 5% share of MAE = 19.0%  (proportional would be 5%)
  Top 5% share of MSE = 42.2%  (proportional would be 5%)
  MSE/MAE sensitivity ratio = 2.23×


#### 8. Classification vs Regression Comparison


In [17]:
print(f"\n{'='*70}")
print("Classification vs Regression — Key Differences")
print(f"{'='*70}")
print(f"""
1. INFORMATION LOSS IN DISCRETIZATION
   The classification target bins mood into 3 classes (low/med/high).
   A prediction of 6.79 and 6.81 fall into different classes despite
   being nearly identical.  The regression target preserves the full
   continuous signal.

2. METRIC BEHAVIOR
   Classification macro-F1 treats all misclassifications equally within
   a class (predicting 'high' for a true 'low' is the same penalty as
   predicting 'medium' for a true 'low').  Regression MAE/MSE are
   sensitive to the magnitude of error — a 2-point miss is penalized
   more than a 0.5-point miss.

3. FEATURE IMPORTANCE STABILITY
   Both tasks rank mood_lag1 as the dominant predictor.  But regression
   may weight rolling features differently since it needs to predict the
   exact value, not just the bin.

4. PRACTICAL IMPLICATION
   For clinical decision support, the regression prediction is more
   useful — it tells clinicians "expect mood of 5.2" rather than just
   "expect low mood".  Classification is useful when the response is
   categorical (e.g., intervene vs. don't).
""")


Classification vs Regression — Key Differences

1. INFORMATION LOSS IN DISCRETIZATION
   The classification target bins mood into 3 classes (low/med/high).
   A prediction of 6.79 and 6.81 fall into different classes despite
   being nearly identical.  The regression target preserves the full
   continuous signal.

2. METRIC BEHAVIOR
   Classification macro-F1 treats all misclassifications equally within
   a class (predicting 'high' for a true 'low' is the same penalty as
   predicting 'medium' for a true 'low').  Regression MAE/MSE are
   sensitive to the magnitude of error — a 2-point miss is penalized
   more than a 0.5-point miss.

3. FEATURE IMPORTANCE STABILITY
   Both tasks rank mood_lag1 as the dominant predictor.  But regression
   may weight rolling features differently since it needs to predict the
   exact value, not just the bin.

4. PRACTICAL IMPLICATION
   For clinical decision support, the regression prediction is more
   useful — it tells clinicians "expect mood of 5

#### 9. Export

In [18]:
# Comparison table
comparison = pd.DataFrame([
    {"model": "naive_baseline", "mae_mean": baseline_models["naive"]["mae"],
     "mse_mean": baseline_models["naive"]["mse"],
     "r2_mean": baseline_models["naive"]["r2"], "n_instances": "–"},
    {"model": "rolling_w5_baseline", "mae_mean": baseline_models["rolling_w5"]["mae"],
     "mse_mean": baseline_models["rolling_w5"]["mse"],
     "r2_mean": baseline_models["rolling_w5"]["r2"], "n_instances": "–"},
    {"model": "xgboost", "mae_mean": xgb_mae, "mse_mean": xgb_mse,
     "r2_mean": xgb_r2, "n_instances": 1394,
     "best_params": str(best_params)},
    {"model": "lstm", "mae_mean": lstm_mae, "mse_mean": lstm_mse,
     "r2_mean": lstm_r2, "n_instances": 660,
     "best_params": str(best_lstm_config)},
])
comparison.to_csv(OUTPUT_DIR / "regression_results.csv", index=False)

pd.Series(best_params).to_csv(OUTPUT_DIR / "xgb_regression_best_params.csv")

print(f"Saved to {OUTPUT_DIR}:")
print(f"  regression_results.csv")
print(f"  regression_errors_xgb.csv   (for Task 5B)")
print(f"  regression_errors_lstm.csv  (for Task 5B)")
print(f"  xgb_regression_feature_importance.csv")
print(f"  xgb_regression_best_params.csv")

print(f"\n✅ Task 4 complete.")

Saved to ..\outputs:
  regression_results.csv
  regression_errors_xgb.csv   (for Task 5B)
  regression_errors_lstm.csv  (for Task 5B)
  xgb_regression_feature_importance.csv
  xgb_regression_best_params.csv

✅ Task 4 complete.


---

**Evaluation setup**: identical walk-forward expanding-window CV as the classification step,
ensuring temporal integrity and direct comparability between classification
and regression results.

**Metric choice**: MAE and MSE are reported jointly because they emphasize
different aspects of prediction quality. MAE measures typical error magnitude
in the original mood scale units. MSE penalizes large errors quadratically,
making it more sensitive to outlier predictions. This distinction is analysed
in depth in the evaluation analysis.

- Both models should substantially beat the naive baseline (which has
  negative R² from b_09, meaning carry-forward is worse than predicting
  the mean).
- Feature importance ranking should be consistent with the classification
  task — mood_lag1 dominant, rolling features for behavioral variables.
- The top-5% error share analysis directly feeds Metric Sensitivity Analysis: MSE's share
  will be ~2× the MAE share, showing the heavy-tail penalty empirically.